# Sign-full ($h_y$) rerun — **v1**: CNN Block-1 + Wilson + N-layer transformer Block-3

Reruns the pure-Y-field sweep $h_y \in \{0.80,\dots,1.20\}$ ($h_x=h_z=0$) at **L=4 and L=6** with the
**v1 architecture** and a **tunable-depth** post-Wilson transformer:

- **Block-1 = the *same* non-invariant CNN as the CNN baseline** (`--channels_noninv 1,16`,
  identity-initialized) → **Wilson nonlinearity** — everything up to and including Wilson is byte-identical
  to the CNN arm.
- **Block-3 = an N-layer factored-attention transformer** replacing the CNN's invariant-CNN+mean.
  **`TF_LAYERS` is a knob** (default **3**; bump to **4** to add depth after the Wilson loop).
- The **encoder stays entirely real** (`float64`); the wavefunction becomes complex **only at the shallow
  readout** $\log\Psi=\sum_\alpha\log\cosh(b_\alpha+w_\alpha\cdot z)$, $b,w\in\mathbb{C}$
  (Viteritti et al., [arXiv:2311.16889](https://arxiv.org/abs/2311.16889)) — `Dense_re(z)+1j·Dense_im(z)`.

**Goal.** Test whether more transformer depth (3–4 layers, d=16) lets v1 **beat the complex CNN** across
the transition — especially at **L=6**, where the d=8 / 2-layer arm (`v1cx8`) lost and blew up
(V-score → 1.0 near $h_y\approx0.95$).

**Baseline:** the committed CNN complex baselines in `results/nqs/` (`..._cnn.json`). **`v1cx8`**
(d=8, 2-layer) is also shown for the depth/width delta. Live to **W&B online** (group `hy_<arm>`).
**Resumable** via Google Drive + skip-if-complete — close the tab and re-run to continue.


## Before you start
1. **Runtime → Change runtime type → GPU** (A100/L4 preferred; T4 works but slow).
2. Have your **GitHub token** (private repo) and **W&B API key** ready.
3. Run top-to-bottom. **After the install cell you may need to restart the runtime** — then resume
   from *Mount Google Drive*.
4. **L=4 runs first** (fast, validates the pipeline) before committing GPU-hours to L=6.


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install the pinned stack

`requirements.txt` pins a **CPU** `jaxlib` (for the NERSC conda env); on Colab we install the
**CUDA 12** build of the same `jax==0.5.2` instead, plus `netket==3.16.1.post1`, `flax==0.10.4`, `wandb`.

⚠️ If pip changes the `jaxlib` already loaded in this session, Colab will ask you to **restart the
runtime** — do it, then resume from the *Mount Google Drive* cell.

In [ ]:
%pip install -q "jax[cuda12]==0.5.2" "jaxlib==0.5.1" netket==3.16.1.post1 flax==0.10.4 wandb
print("\nInstall finished. If a 'restart runtime' notice appeared, do Runtime > Restart session,")
print("then continue from the 'Mount Google Drive' cell (do NOT re-run this install cell).")

In [ ]:
import jax, jaxlib, netket, flax
print("jax", jax.__version__, "| jaxlib", jaxlib.__version__,
      "| netket", netket.__version__, "| flax", flax.__version__)
print("devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), \
    "No GPU visible to JAX — set Runtime>GPU and/or re-check the jax[cuda12] install."
print("OK: GPU visible.")

## 3. Mount Google Drive  *(resume point after any runtime restart)*
Results and the cloned repo live under `MyDrive/2D-TC-colab/` so they survive disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone (or update) the repo into Drive
Private repo → paste a GitHub token when prompted. It is used only for the clone, then stripped from
the stored remote (not saved to Drive).

In [ ]:
import os, getpass, subprocess

DRIVE_ROOT = "/content/drive/MyDrive/2D-TC-colab"   # persists across sessions
BRANCH     = "v3-complex-readout"                    # has the transformer code + CNN baselines
REPO_DIR   = os.path.join(DRIVE_ROOT, "2D-TC")
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    tok = getpass.getpass("GitHub token (repo read access): ")
    url = f"https://{tok}@github.com/SanzharBissenali/2D-TC.git"
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                    "https://github.com/SanzharBissenali/2D-TC.git"], check=True)
    del tok, url
else:
    print("repo already present; trying a fast-forward pull (harmless if it fails)")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("repo:", REPO_DIR, "| branch:", head)
assert os.path.exists("model/transformer_block.py"), "transformer code missing — wrong branch?"
print("OK: transformer code present.")

## 5. Log in to Weights & Biases (online)
Key from [wandb.ai/authorize](https://wandb.ai/authorize). Stored in `~/.netrc`, which the training
subprocesses inherit, so runs stream **live** to the dashboard.

In [ ]:
import os, getpass, wandb

os.environ["WANDB_MODE"] = "online"          # Colab has internet -> truly live
wandb.login(key=getpass.getpass("W&B API key (wandb.ai/authorize): "))

WANDB_PROJECT = "2d-tc"
WANDB_ENTITY  = ""     # optional: your wandb team/username; "" = your default entity
print("W&B online. project =", WANDB_PROJECT, "| entity =", WANDB_ENTITY or "(default)")

## 6. Sweep configuration
**`TF_LAYERS`** = number of transformer blocks after Wilson (the knob you asked for). **`D_MODEL`** =
transformer width (drop to 8 to ~halve time). `SIM_TIME=3.0` = 300 TDVP steps (baselines used 200; the
extra steps are the L=6 "rescue" — the variational comparison stays valid either way).

In [ ]:
D_MODEL   = 16      # transformer width; D_MODEL % TF_HEADS == 0
TF_LAYERS = 3       # <<< KNOB: transformer blocks AFTER the Wilson nonlinearity (try 4)
TF_HEADS  = 2
LX_LIST   = [4, 6]                                          # L=4 first, then L=6
HY_LIST   = [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20]
SEED      = 0
SIM_TIME  = 3.0                                             # 300 steps (dt=0.01)

ARM = f"v1cx{D_MODEL}n{TF_LAYERS}"                          # e.g. v1cx16n3 -> in the filenames
ARM_FLAGS = (
    f"--symmetric_block transformer "                       # v1: CNN Block-1 kept, TF replaces Block-3
    f"--tf_layers {TF_LAYERS} --tf_dmodel {D_MODEL} --tf_heads {TF_HEADS} "
    f"--tf_ffn_mult 2 --tf_activation relu "
    f"--tf_complex_output --diag_shift 1e-3 --lr_schedule cosine --lr_final_frac 0.25"
)
WANDB_GROUP = f"hy_{ARM}"
print("arm:", ARM, "|", len(LX_LIST) * len(HY_LIST), "points | flags:", ARM_FLAGS)

## 7. Run the sweep (resumable, live to W&B)
Skip-if-complete: re-run this cell after any disconnect to continue. Interrupt anytime — the current
point resumes next run.

In [ ]:
import os, subprocess, sys

REPO   = os.getcwd()
OUTDIR = os.path.join(REPO, "results", "nqs")
os.makedirs(OUTDIR, exist_ok=True)
env = dict(os.environ, PYTHONPATH=REPO, WANDB_MODE="online", WANDB_DIR=REPO)

def is_complete(base, L):
    return subprocess.run([sys.executable, os.path.join(REPO, "scripts", "is_complete.py"),
                           base, str(L)]).returncode == 0

for L in LX_LIST:
    for hy in HY_LIST:
        jobid = f"L{L}_hx0.00_hy{hy:.2f}_{ARM}"
        base  = os.path.join(OUTDIR, f"G-equiv_1_{jobid}")
        if is_complete(base, L):
            print(f"== skip {jobid} (complete) =="); continue
        print(f"\n{'='*70}\n RUN {jobid}   (sim_time={SIM_TIME})\n{'='*70}", flush=True)
        entity = f" --wandb_entity {WANDB_ENTITY}" if WANDB_ENTITY else ""
        cmd = (
            f"cd {OUTDIR} && PYTHONPATH={REPO} python {REPO}/main.py "
            f"--outindex 1 --jobid {jobid} "
            f"--Lx {L} --hx 0.0 --hy {hy:.2f} --hz 0.0 "
            f"--dt 0.01 --diag_shift 6e-5 --sim_time {SIM_TIME} --seed {SEED} "
            f"--architecture Combo --channels_noninv 1,16 --channels_inv 16,8,1 --kernel_size 2 "
            f"--n_samples_fin 8192 --use_custom_sampler "
            f"{ARM_FLAGS} "
            f"--wandb --wandb_project {WANDB_PROJECT} --wandb_group {WANDB_GROUP}{entity}"
        )
        rc = subprocess.run(["bash", "-lc", cmd], env=env).returncode
        print(f"===== {jobid} exit {rc} =====", flush=True)

print("\nSweep pass complete. Re-run to resume any unfinished points.")

## 8. Analysis — v1 N-layer transformer vs complex CNN
`dE = E_transformer − E_CNN`; **negative ⇒ transformer wins** (lower variational energy).

In [ ]:
import json, os, statistics as st
import numpy as np, matplotlib.pyplot as plt

OUTDIR = os.path.join(os.getcwd(), "results", "nqs")
HY = [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20]; TAIL = 10

def _c(x):
    return complex(x) if not isinstance(x, dict) else complex(x.get("real", np.nan), x.get("imag", 0.0))
def load(tag):
    p = os.path.join(OUTDIR, f"G-equiv_1_{tag}.json")
    if not os.path.exists(p): return None
    e = [_c(x).real for x in json.load(open(p)).get("energy", [])]
    v = [_c(x).real for x in json.load(open(p)).get("Vscore", [])]
    return None if not e else dict(E=st.median(e[-TAIL:]), V=(st.median(v[-TAIL:]) if v else np.nan))

for L in [4, 6]:
    print(f"\n=== L={L}:  {ARM} vs CNN   (dE<0 => transformer wins) ===")
    print(f"{'hy':>5} {'E_cnn':>12} {'E_'+ARM:>13} {'dE':>10} {'V_cnn':>9} {'V_tf':>9} {'E_v1cx8':>12} {'win':>5}")
    for hy in HY:
        c  = load(f"L{L}_hx0.00_hy{hy:.2f}_cnn")
        t  = load(f"L{L}_hx0.00_hy{hy:.2f}_{ARM}")
        t1 = load(f"L{L}_hx0.00_hy{hy:.2f}_v1cx8")
        if not c: print(f"{hy:>5.2f}  (no CNN baseline)"); continue
        if not t: print(f"{hy:>5.2f} {c['E']:>12.4f} {'(pending)':>13}"); continue
        dE = t["E"] - c["E"]; e1 = f"{t1['E']:>12.4f}" if t1 else f"{'--':>12}"
        print(f"{hy:>5.2f} {c['E']:>12.4f} {t['E']:>13.4f} {dE:>+10.4f} {c['V']:>9.1e} {t['V']:>9.1e} {e1} {('TF' if dE<0 else 'CNN'):>5}")

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for j, L in enumerate([4, 6]):
    hyv, dE, Vc, Vt = [], [], [], []
    for hy in HY:
        c = load(f"L{L}_hx0.00_hy{hy:.2f}_cnn"); t = load(f"L{L}_hx0.00_hy{hy:.2f}_{ARM}")
        if c and t: hyv.append(hy); dE.append(t["E"]-c["E"]); Vc.append(c["V"]); Vt.append(t["V"])
    a = ax[0, j]; a.axhline(0, color="k", lw=0.8); a.plot(hyv, dE, "o-", color="crimson")
    a.fill_between(hyv, dE, 0, where=[d < 0 for d in dE], alpha=0.2, color="green")
    a.set(title=f"L={L}: dE = E({ARM}) - E(CNN)", xlabel="$h_y$", ylabel="dE")
    b = ax[1, j]; b.plot(hyv, Vc, "s--", color="gray", label="CNN"); b.plot(hyv, Vt, "o-", color="crimson", label=ARM)
    b.set(title=f"L={L}: V-score", xlabel="$h_y$", ylabel="V-score", yscale="log"); b.legend()
fig.tight_layout(); plt.show()

## Runtime notes
- Real encoder ⇒ compile is far cheaper than a fully-complex CNN, but L=6's first step still takes a
  few minutes (then fast). **L=4 first, then L=6.**
- Sessions: free Colab ~90 min idle / ~12 h; Pro longer. Drive + skip-if-complete ⇒ just re-run to resume.
- **Too slow?** set `D_MODEL=8`, and/or shrink `HY_LIST` to `{0.90,0.95,1.00,1.05,1.10}` first.
- On wandb.ai (project `2d-tc`, group `hy_<arm>`): watch energy / V-score / `phase_circ_var` / QGT cond.
  A V-score stuck ~1.0 = diverged → bump `--diag_shift` to `3e-3` for that L.
- Results: `MyDrive/2D-TC-colab/2D-TC/results/nqs/` — commit to git from a laptop later if wanted.
